In [0]:
'''
Notebook    : 41_bv_process
Author      : srinivas 
Date        : 7thAug 2025
Description : This notebook is used to process the data
              i) create dataframe on raw folder 
            ii) add audit columns(load date)
            iii) overwrite the bronze table
            iv)  from bronze to silver apply the transformations (if any) and load 
                        1) reviews    ---> append 
                        2) categories --> overwrite
                         3) products   --> scd type1
            v) move the raw files into archive folder

'''

In [0]:
#table_name = 'reviews'
table_name = dbutils.widgets.get("tbl_nm")

In [0]:
external_loc = 'abfss://srinivasbatch60@bvsrinivasbatch60.dfs.core.windows.net'
raw_path     =  external_loc+'/raw'
bv_raw_path  =  raw_path+ '/BV'
tbl_raw_path =  bv_raw_path+'/'+ table_name

bronze_schema = 'bronze'
bronze_tbl_nm = table_name+'_stg'

silver_tbl_path = external_loc+'/silver/'+table_name
silver_schema = 'silver'
silver_tbl_nm = table_name

archive_path     =  external_loc+'/archive'
bv_archive_path  =  archive_path+ '/BV'
tbl_archive_path =  bv_archive_path+'/'+ table_name
 


In [0]:
dbutils.fs.mkdirs(silver_tbl_path)
dbutils.fs.mkdirs(tbl_archive_path)

True

In [0]:
tbl_raw_path

'abfss://srinivasbatch60@bvsrinivasbatch60.dfs.core.windows.net/raw/BV/reviews'

In [0]:
dbutils.fs.ls(tbl_raw_path)

[FileInfo(path='abfss://srinivasbatch60@bvsrinivasbatch60.dfs.core.windows.net/raw/BV/reviews/reviews_01082025.json', name='reviews_01082025.json', size=782, modificationTime=1754448263000)]

In [0]:
dbutils.fs.ls('abfss://srinivasbatch60@bvsrinivasbatch60.dfs.core.windows.net/raw')

[FileInfo(path='abfss://srinivasbatch60@bvsrinivasbatch60.dfs.core.windows.net/raw/BV/', name='BV/', size=0, modificationTime=1754447705000)]

In [0]:
df = spark.read.format("json").load(tbl_raw_path)

df.display()

cust_id,prod_id,rating,review_id,review_text,reviewdate
C1,PROD001,5,REVW001,product is not good,2025-08-01
C2,PROD002,4,REVW002,Nice product,2025-08-01
C3,PROD003,3,REVW003,worthable product,2025-08-01
C4,PROD006,2,REVW004,use less product,2025-08-01
C1,PROD006,1,REVW005,product is not good,2025-08-01
C2,PROD001,5,REVW006,Nice product,2025-08-01


In [0]:
%sql 
show catalogs;

catalog
devcatalog_srinivas_batch60
hive_metastore
pvrcloudtechadbworkspace
samples
system


In [0]:
%sql 
use catalog devcatalog_srinivas_batch60;

In [0]:
%sql

show schemas

databaseName
bronze
default
information_schema
silver


<h3>Load data from raw to Bronze</h3>

In [0]:
df = spark.read.format("json").load(tbl_raw_path)

In [0]:
from datetime import datetime 

dt = datetime.now().strftime("%d%m%Y")
print(dt)

07082025


In [0]:
from pyspark.sql.functions import lit

In [0]:
df = df.withColumn("load_date",lit(dt))
df.display()


cust_id,prod_id,rating,review_id,review_text,reviewdate,load_date
C1,PROD001,5,REVW001,product is not good,2025-08-01,07082025
C2,PROD002,4,REVW002,Nice product,2025-08-01,07082025
C3,PROD003,3,REVW003,worthable product,2025-08-01,07082025
C4,PROD006,2,REVW004,use less product,2025-08-01,07082025
C1,PROD006,1,REVW005,product is not good,2025-08-01,07082025
C2,PROD001,5,REVW006,Nice product,2025-08-01,07082025


In [0]:
id = 101
name = 'srinivas'

str1 = f"person id is {id} and name is {name}"
print(str1)
#output
#person id is 101 and name is srininvas

person id is 101 and name is srinivas


In [0]:
df.write.format("Delta").mode("overwrite").saveAsTable(f"{bronze_schema}.{bronze_tbl_nm}")

In [0]:
%sql 

select * from bronze.reviews_stg

cust_id,prod_id,rating,review_id,review_text,reviewdate,load_date
C1,PROD001,5,REVW001,product is not good,2025-08-01,07082025
C2,PROD002,4,REVW002,Nice product,2025-08-01,07082025
C3,PROD003,3,REVW003,worthable product,2025-08-01,07082025
C4,PROD006,2,REVW004,use less product,2025-08-01,07082025
C1,PROD006,1,REVW005,product is not good,2025-08-01,07082025
C2,PROD001,5,REVW006,Nice product,2025-08-01,07082025


In [0]:
%sql 

desc formatted bronze.reviews_stg;

col_name,data_type,comment
cust_id,string,null
prod_id,string,null
rating,bigint,null
review_id,string,null
review_text,string,null
reviewdate,string,null
load_date,string,null
,,
# Delta Statistics Columns,,
Column Names,"prod_id, review_id, load_date, review_text, rating, cust_id, reviewdate",


<h3>load data from bronze to silver</h3>

In [0]:
if table_name == 'reviews':
    df.write.format("Delta").mode("append").save(silver_tbl_path)
elif table_name == 'categories':
    df.write.format("Delta").mode("overwrite").save(silver_tbl_path)


In [0]:
silver_tbl_path

'abfss://srinivasbatch60@bvsrinivasbatch60.dfs.core.windows.net/silver/reviews'

In [0]:
qry = f'''
create table if not exists {silver_schema}.{silver_tbl_nm}
using delta 
location '{silver_tbl_path}'
'''
spark.sql(qry)

DataFrame[]

In [0]:
df = spark.sql(f'select * from silver.{silver_tbl_nm}')
df.display()

cust_id,prod_id,rating,review_id,review_text,reviewdate,load_date
C1,PROD001,5,REVW001,product is not good,2025-08-01,07082025
C2,PROD002,4,REVW002,Nice product,2025-08-01,07082025
C3,PROD003,3,REVW003,worthable product,2025-08-01,07082025
C4,PROD006,2,REVW004,use less product,2025-08-01,07082025
C1,PROD006,1,REVW005,product is not good,2025-08-01,07082025
C2,PROD001,5,REVW006,Nice product,2025-08-01,07082025


In [0]:
dbutils.fs.ls(tbl_archive_path)

[]

In [0]:
files_lst = dbutils.fs.ls(tbl_raw_path)

for file in files_lst: 
    dbutils.fs.mv(file.path,tbl_archive_path)

In [0]:
dbutils.fs.ls(tbl_archive_path)

[FileInfo(path='abfss://srinivasbatch60@bvsrinivasbatch60.dfs.core.windows.net/archive/BV/reviews/reviews_01082025.json', name='reviews_01082025.json', size=0, modificationTime=1754534764000)]